# Benchmarking the Synthetic-Evidence-Image-Corpus with DetectZoo

This notebook runs [DetectZoo](https://github.com/sadjadeb/DetectZoo) image detectors over your two-class
corpus and reports a full benchmark: **accuracy, precision, recall, F1, TPR/FPR, ROC-AUC and PR-AUC.**

| Class | Label | Folder |
|---|---|---|
| Authentic (real) | `0` | `...\cifar-synthetic-evidence-corpus\Source-Images` |
| Manipulated (fake) | `1` | `...\cifar-synthetic-evidence-corpus\Manipulated-Images` |

### Notes before you run
- **ROC-AUC / PR-AUC are the fairest cross-detector comparison.** Accuracy, precision, recall, F1, TPR
  and FPR depend on each detector's *built-in default threshold*, which is not tuned to your corpus, so
  treat them as indicative. The AUC metrics are threshold-free.
- **Class balance.** If `Source-Images` and `Manipulated-Images` are very different sizes, accuracy will
  be skewed toward the larger class — lean on PR-AUC / per-class counts in that case.
- **Detector scope.** These models are trained mostly on *fully synthesised* images. On local edits /
  splices / retouching their numbers may be lower than on GAN/diffusion images. That's a property of the
  detectors, not your labels.
- **First run downloads model weights** (needs internet). **Use a GPU** if you have one.
- DetectZoo is on **TestPyPI** for now (anonymized `detectzoo-anon`); import name is still `detectzoo`.


## 1. Install DetectZoo

Run once; skip if already installed.

In [1]:
# import sys

# !{sys.executable} -m pip install --index-url https://test.pypi.org/simple/ \
#     --extra-index-url https://pypi.org/simple/ detectzoo-anon
# !{sys.executable} -m pip install -q pandas tqdm

## 2. Configuration

In [2]:
from pathlib import Path
import pandas as pd

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

CORPUS = Path(r"C:\Users\kelly.mcconvey\Documents\GitHub\cifar-synthetic-evidence-corpus")
REAL_DIR = CORPUS / "Source-Images"        # authentic  -> label 0
FAKE_DIR = CORPUS / "Manipulated-Images"   # manipulated -> label 1

print("Device     :", DEVICE)
print("Real  found:", REAL_DIR.exists(), "->", REAL_DIR)
print("Fake  found:", FAKE_DIR.exists(), "->", FAKE_DIR)

Device     : cpu
Real  found: True -> C:\Users\kelly.mcconvey\Documents\GitHub\cifar-synthetic-evidence-corpus\Source-Images
Fake  found: True -> C:\Users\kelly.mcconvey\Documents\GitHub\cifar-synthetic-evidence-corpus\Manipulated-Images


## 3. Build the labelled dataset

DetectZoo ships `BaseDataset.from_directory(real, fake)`, but it only reads the **top level** of each folder and doesn't tag the modality. The small subclass below recurses into subfolders, keeps the filename in metadata, and sets `modality="image"` so the printed table uses the image column set.

In [3]:
from detectzoo.datasets.base import BaseDataset, DatasetItem

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp", ".gif"}

class ImageFolderDataset(BaseDataset):
    """Two-folder image dataset (recursive). real -> 0, fake -> 1."""
    name = "evidence_corpus"
    modality = "image"

    def __init__(self, real_dir, fake_dir, **kw):
        super().__init__(**kw)
        self.real_dir, self.fake_dir = Path(real_dir), Path(fake_dir)

    def _imgs(self, d):
        return sorted(p for p in Path(d).rglob("*") if p.suffix.lower() in IMG_EXTS)

    def _load_all(self):
        items = [DatasetItem(str(p), 0, {"source": "real", "file": p.name})
                 for p in self._imgs(self.real_dir)]
        items += [DatasetItem(str(p), 1, {"source": "fake", "file": p.name})
                  for p in self._imgs(self.fake_dir)]
        return items

dataset = ImageFolderDataset(REAL_DIR, FAKE_DIR)
items = dataset.load()
n_real = sum(it.label == 0 for it in items)
n_fake = sum(it.label == 1 for it in items)
print(f"Real (0): {n_real}    Fake (1): {n_fake}    Total: {len(items)}")

Real (0): 2000    Fake (1): 2318    Total: 4318


## 4. Available image detectors

See the [methods table](https://github.com/sadjadeb/DetectZoo/blob/main/METHODS_AND_MODELS.md) for details. Solid starting set: `univfd` (CLIP, generalizes well), `cnnspot` (classic CNN artifacts), `aeroblade` (training-free diffusion), `patchcraft`, `npr_deepfake`.

In [4]:
from detectzoo import list_detectors, load_detector
print(list_detectors("image"))

['aeroblade', 'aide', 'c2p_clip', 'cnnspot', 'cospy', 'cospy_sd_v1_4', 'd3', 'drct', 'fatformer', 'freqnet', 'ladeda', 'lgrad', 'manifold_bias', 'npr_deepfake', 'patchcraft', 'safe', 'univfd']


## 5. Run the benchmark

This does a **single** evaluation pass: every detector scores every image once, then metrics are computed and the results saved to JSON. `save_scores=True` keeps per-image scores for the analysis in step 7. `unload_between=True` frees each model's weights before loading the next (kinder to GPU memory). The first run of each detector downloads its weights.

In [5]:
from detectzoo.benchmarks import BenchmarkEvaluator

DETECTORS = ["d3","univfd","drct","cnnspot"]

detector_objs = [load_detector(name, device=DEVICE) for name in DETECTORS]

out_dir = CORPUS / "detectzoo_results"
out_dir.mkdir(exist_ok=True)

evaluator = BenchmarkEvaluator(dataset)
results = evaluator.run_and_save(
    detector_objs,
    output_path=str(out_dir / "benchmark.json"),
    save_scores=True,
    unload_between=True,
    meta={"real_dir": str(REAL_DIR), "fake_dir": str(FAKE_DIR),
          "n_real": n_real, "n_fake": n_fake, "device": DEVICE},
)
print("Done. JSON saved to", out_dir / "benchmark.json")

[2026-06-10 09:25:44] detectzoo.datasets._download INFO: Downloading https://github.com/BigAandSmallq/D3/raw/main/ckpt/classifier.pth


  100.0%  (12,607,488 / 12,606,728 bytes)


c:\Users\kelly.mcconvey\AppData\Local\Programs\Python\Python313\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
[2026-06-10 09:25:50] detectzoo.datasets._download INFO: Downloading https://github.com/WisconsinAIVision/UniversalFakeDetect/raw/main/pretrained_weights/fc_weights.pth
INFO:detectzoo.datasets._download:Downloading https://github.com/WisconsinAIVision/UniversalFakeDetect/raw/main/pretrained_weights/fc_weights.pth


  100.0%  (8,192 / 4,083 bytes)


[2026-06-10 09:25:52] detectzoo.datasets._download INFO: Downloading https://modelscope.cn/datasets/BokingChen/DRCT-2M/resolve/master/pretrained.zip
INFO:detectzoo.datasets._download:Downloading https://modelscope.cn/datasets/BokingChen/DRCT-2M/resolve/master/pretrained.zip


  100.0%  (4,178,796,544 / 4,178,790,241 bytes)


c:\Users\kelly.mcconvey\AppData\Local\Programs\Python\Python313\Lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(
[2026-06-10 09:33:24] detectzoo.datasets._download INFO: Downloading https://www.dropbox.com/s/2g2jagq2jn1fd0i/blur_jpg_prob0.5.pth?dl=1
INFO:detectzoo.datasets._download:Downloading https://www.dropbox.com/s/2g2jagq2jn1fd0i/blur_jpg_prob0.5.pth?dl=1


  100.0%  (282,443,776 / 282,442,597 bytes)


d3:  47%|████▋     | 2008/4318 [32:46<38:34,  1.00s/it] c:\Users\kelly.mcconvey\AppData\Local\Programs\Python\Python313\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (114468680 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
d3:  59%|█████▉    | 2566/4318 [42:16<28:46,  1.01it/s]c:\Users\kelly.mcconvey\AppData\Local\Programs\Python\Python313\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (94080000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
univfd:  47%|████▋     | 2008/4318 [16:42<18:37,  2.07it/s]c:\Users\kelly.mcconvey\AppData\Local\Programs\Python\Python313\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (114468680 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
univfd:  59%|█████▉    | 2566/4318 [21:30<14:24,  2.03it/s]c:\Users\kelly.mcconvey\AppD

Done. JSON saved to C:\Users\kelly.mcconvey\Documents\GitHub\cifar-synthetic-evidence-corpus\detectzoo_results\benchmark.json


## 6. Results table

Built from the metrics `run_and_save` returned (no second pass). Sorted by ROC-AUC.

In [6]:
METRIC_COLS = ["accuracy", "precision", "recall", "f1",
               "tpr", "fpr", "roc_auc", "pr_auc", "avg_precision", "n_samples"]

rows = [{"detector": name, **{c: m.get(c) for c in METRIC_COLS}}
        for name, m in results.items()]
bench_df = (pd.DataFrame(rows)
              .sort_values("roc_auc", ascending=False)
              .reset_index(drop=True))

bench_df.to_csv(out_dir / "benchmark_summary.csv", index=False)
bench_df.round(4)

,detector,accuracy,precision,recall,f1,tpr,fpr,roc_auc,pr_auc,avg_precision,n_samples
0,d3,0.5697,0.6423,0.4478,0.5277,0.4478,0.2890,0.5853,0.6510,0.6511,4318
1,univfd,0.5225,0.6604,0.2274,0.3383,0.2274,0.1355,0.5784,0.5930,0.5934,4318
2,drct,0.4817,0.5217,0.4154,0.4625,0.4154,0.4415,0.4937,0.5784,0.5786,4318
3,cnnspot,0.5841,0.9379,0.2412,0.3837,0.2412,0.0185,0.4760,0.6444,0.6445,4318


## 7. (Optional) Per-image scores — which manipulations slip through

`save_scores=True` stored a score per image in dataset order. This joins those scores back to filenames so you can see the manipulated images that scored *lowest* (least detectable) for a chosen detector. Higher score = more likely AI/manipulated.

In [10]:
DETECTOR = "cnnspot"   # pick any name from DETECTORS

files  = [it.metadata["file"] for it in items]
labels = [it.label for it in items]
scores = [s["score"] for s in results[DETECTOR]["samples"]]

per_image = pd.DataFrame({"file": files, "true_label": labels, "score": scores})
per_image["class"] = per_image["true_label"].map({0: "real", 1: "fake"})
per_image.to_csv(out_dir / f"per_image_{DETECTOR}.csv", index=False)

# Manipulated images this detector scored lowest (hardest to catch):
print(f"Hardest-to-detect manipulations for '{DETECTOR}':")
per_image[per_image.true_label == 1].nsmallest(10, "score")[["file", "score"]]

Hardest-to-detect manipulations for 'cnnspot':


,file,score
3917,d15a226a-1645-5d0d-b640-fec41a2668d4-TRN-RCT-T...,5.901410e-17
2535,3d3730e8-5258-5763-acd7-7fd038678135-TRN-RCT-T...,1.301731e-15
3632,b2857f05-996a-54ae-b61f-830a0e19e4cc-TST-DOC-T...,4.009399e-15
3716,bb5a4bc8-29d4-5e7a-b21a-52365355f93d-TRN-DOC-T...,3.170427e-14
3685,b862757c-33b5-5920-bffb-c829ca72fa8e-TRN-RCT-T...,5.924968e-14
2657,48feb5e5-b344-56f7-9783-5f0cc6d1236e-TRN-RCT-T...,7.414430e-14
2663,49968d36-0880-5278-a7b8-98d485748c50-TRN-RCT-T...,7.722768e-14
2509,3a9bfe14-e932-50b4-a73c-5a2ec3875c15-TRN-RCT-T...,1.527946e-13
2392,2ccdb4f0-9542-5ab3-a171-7f9c9f276219-TST-RCT-T...,1.801704e-13
4184,f19f5aff-c9aa-579c-8cbc-ef6ef2cc9977-TRN-DOC-T...,2.087810e-13
